In [4]:
from fastmcp import FastMCP
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import uuid

In [5]:
mcp = FastMCP("RAG on MCP")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
# embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

db_name = "./chroma_vector_store"

In [6]:
@mcp.tool
def ingest_document(file_path:str):
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
        chunks = splitter.split_documents(docs)

        metadatas = [{"source": file_path, "page": doc.metadata.get("page", 0)} for doc in chunks]
        ids = [str(uuid.uuid4()) for _ in chunks]


        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            metadatas=metadatas,
            ids=ids,
            persist_directory=db_name
        )
        return f"Success: Ingested {len(chunks)} chunks from {file_path} into the vector database."
    except Exception as e:
        return f"Error ingesting file: {str(e)}"

In [7]:
@mcp.tool
def fetch_documents(query:str):
    try:
        vector_db = Chroma(
            embedding_function = embeddings,
            persist_directory = db_name
        )
        results = vector_db.similarity_search(query=query, k=5)

        if not results:
            return "No relevant documents found in the database."
        
        formatted_results = "Here are the top matching documents from the database:\n\n"
        for i, doc in enumerate(results):
            # Extract metadata if available (PyPDFLoader usually adds 'source' and 'page')
            source = doc.metadata.get("source", "Unknown file")
            page = doc.metadata.get("page", "Unknown page")
                
            formatted_results += f"--- Match {i+1} (Source: {source}, Page: {page}) ---\n"
            formatted_results += f"{doc.page_content}\n\n"
                
        return formatted_results
        
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"


In [8]:
fetch_documents("what is agentic ai")

/tmp/ipykernel_13814/1947122300.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


'Here are the top matching documents from the database:\n\n--- Match 1 (Source: data/agenticAi.pdf, Page: 3) ---\nUnlike traditional AI, which relies on predefined rules and human oversight, agentic AI exhibits goal-driven behavior, adaptability, and\nthe ability to interact dynamically with its environment.\nThese systems leverage AI agents, which are machine learning models capable of mimicking human decision-making. They can\nperform tasks independently, coordinate with other agents in multi-agent systems, and adapt their strategies based on feedback and\nlearning.\nKey Characteristics of Agentic AI (with Examples)\n🎯  Running Example\nGoal:\nPlan a 3-day trip to Bengaluru within a ₹ 20,000 budget and complete all bookings autonomously.\nThis same example is used across all characteristics.\n1. Autonomy\nWhat it means\nThe AI agent can make decisions and take actions independently without needing step-by-step human instructions.\nExample\nThe agent:\nSearches transport options\nChec

In [9]:
if __name__ == "__main__":
    mcp.run()

RuntimeError: Already running asyncio in this thread